# 02 - Transformación y limpieza de datos
## Contratos Menores - Ayuntamiento de Vilagarcía de Arousa

Este notebook toma el fichero crudo descargado en el paso anterior y produce un dataset limpio, tipado y listo para cargar en Snowflake.

**Entrada:** `data/raw/contratos_menores.xlsx`  
**Salida:** `data/processed/contratos_limpios.csv`

---

### Decisiones y supuestos documentados

| # | Decisión | Justificación |
|---|---|---|
| 1 | Se eliminan `Unnamed: 0` y `Lista de lotes` | 100% y 99.8% nulas respectivamente. Sin valor analítico. |
| 2 | `Código CPV` (96% nulo) se conserva tal cual | Podría ser útil para análisis futuros; se anota el alto % de nulos. |
| 3 | Se separa `Contratistas` en `nif_contratista` y `nombre_contratista` | El campo mezcla CIF/NIF con nombre en un solo string (`"B27188317 - EXCAVACIONES J.CARREIRA"`). |
| 4 | Los importes (strings con formato español `"1.234,56"`) se convierten a `float` | Requiere quitar los puntos de miles y sustituir la coma decimal por punto. |
| 5 | Los nombres de columna se normalizan a snake_case sin tildes ni espacios | Facilita el trabajo en SQL y Python. |

## 0 — Importaciones y rutas

In [ ]:
import re
import pandas as pd
from pathlib import Path

RUTA_RAW       = Path("../data/raw/contratos_menores.xlsx")
RUTA_PROCESADO = Path("../data/processed/contratos_limpios.csv")

print(f"Leyendo: {RUTA_RAW}")

## 1 — Cargar el fichero crudo

In [ ]:
df = pd.read_excel(RUTA_RAW, dtype=str)  # dtype=str para no perder ceros a la izquierda en CIFs

print(f"Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")
df.head(3)

## 2 — Eliminar columnas sin valor

In [ ]:
COLUMNAS_BASURA = ["Unnamed: 0", "Lista de lotes"]
df = df.drop(columns=COLUMNAS_BASURA)

print(f"Columnas tras limpieza: {df.shape[1]}")
print(df.columns.tolist())

## 3 — Renombrar columnas a snake_case

In [ ]:
RENOMBRAR = {
    "Entidad Contratante":                               "entidad_contratante",
    "Número de Referencia del Contrato":                  "num_referencia",
    "Objeto del Contrato":                               "objeto_contrato",
    "Valor estimado del contrato (en euros)":             "valor_estimado_eur",
    "Presupuesto base sin impuestos (en euros)":          "presupuesto_base_eur",
    "Plazo ejecución (en meses)":                        "plazo_meses",
    "Código CPV del objeto del contrato":                 "codigo_cpv",
    "Tipo de contratación":                              "tipo_contratacion",
    "Tipo de Contrato":                                  "tipo_contrato",
    "Tipo de procedimiento":                             "tipo_procedimiento",
    "Sistema de contratación":                           "sistema_contratacion",
    "Tramitación":                                       "tramitacion",
    "Fecha formalización":                               "fecha_formalizacion",
    "Importe total ofertado (sin impuestos) (en euros)": "importe_sin_iva_eur",
    "Importe total ofertado (con impuestos) (en euros)": "importe_con_iva_eur",
    "URL a la licitación específica del expediente":      "url_licitacion",
    "Observaciones":                                     "observaciones",
    "Contratistas":                                      "contratistas_raw",
}

df = df.rename(columns=RENOMBRAR)
print(df.columns.tolist())

## 4 — Limpiar e importar la columna de contratistas

Formato original: `"B27188317 - EXCAVACIONES J.CARREIRA, S.L."`  
Se separa por el primer ` - ` en NIF/CIF y nombre.

In [ ]:
def separar_contratista(texto):
    """Devuelve (nif, nombre) a partir de 'NIF - NOMBRE'."""
    if pd.isna(texto) or not isinstance(texto, str):
        return pd.NA, pd.NA
    partes = texto.split(" - ", maxsplit=1)
    if len(partes) == 2:
        return partes[0].strip(), partes[1].strip()
    return pd.NA, texto.strip()

df[["nif_contratista", "nombre_contratista"]] = (
    df["contratistas_raw"]
    .apply(lambda x: pd.Series(separar_contratista(x)))
)

# Verificación
print("Ejemplos de separación:")
df[["contratistas_raw", "nif_contratista", "nombre_contratista"]].head(5)

## 5 — Convertir importes de string español a float

Formato español: `"1.234,56"` → `1234.56`  
Proceso: quitar puntos de miles, sustituir coma decimal por punto, convertir a float.

In [ ]:
def euros_a_float(valor):
    """Convierte un string de importe en formato español a float."""
    if pd.isna(valor) or str(valor).strip() in ("", "nan"):
        return pd.NA
    limpio = str(valor).strip().replace(".", "").replace(",", ".")
    try:
        return float(limpio)
    except ValueError:
        return pd.NA

COLS_IMPORTE = ["valor_estimado_eur", "presupuesto_base_eur", "importe_sin_iva_eur", "importe_con_iva_eur"]

for col in COLS_IMPORTE:
    df[col] = df[col].apply(euros_a_float).astype("Float64")

print("Tipos resultantes:")
print(df[COLS_IMPORTE].dtypes)
print("\nEjemplos:")
df[COLS_IMPORTE].head()